# Layer Normalization - Ba, Kiros & Hinton (2016)

*Layer Normalization*, arXiv:1607.06450 (July 2016).

Paper: `~/personal/knowledge/projects/nlp-to-llm-evolution/sources/ba-2016-layernorm.pdf`

Tier: Light (2 working days). Started 2026-11-10.


## Pain point: before this paper

Batch Normalization (Ioffe & Szegedy, 2015) normalizes each neuron $i$ **across the mini-batch** (`dim = 0`, shape `(H,)`):
- In variable-length RNNs (e.g. batch of 3 sentences with lengths `3, 5, 8`), unmasked `<pad>` tokens corrupt the batch and running statistics.
- Even if `<pad>` positions are masked out and separate statistics $\mu_t, \sigma_t$ are stored per time step $t$, tail steps ($t = 6, 7, 8$) have only 1 active sentence (`sigma_t = 0`), and small mini-batches have noisy statistics.
- At test time, if a sentence arrives with length $> 8$ (longer than any training sequence), no stored running $\mu_t, \sigma_t$ exist for those later steps.


## The idea: normalizing within a single training case

Given pre-activations $a_t \in \mathbb{R}^{B \times H}$, **Layer Normalization** transposes BatchNorm by computing mean and standard deviation **row-wise across all $H$ hidden units of each individual example** (`dim = -1`, shape `(B, 1)`):

$$\mu^l = \frac{1}{H}\sum_{i=1}^H a_i^l, \qquad \sigma^l = \sqrt{\frac{1}{H}\sum_{i=1}^H (a_i^l - \mu^l)^2}, \qquad \hat{a}^l = \frac{g}{\sigma^l} \odot (a^l - \mu^l) + b$$

- **Statistics $\mu, \sigma$ have shape `(B, 1)`**: computed on the fly per example at each time step $t$. Train and test use the exact same deterministic formula, `B = 1` works identically to `B = 64`, and `<pad>` in one sequence never touches another sequence.
- **Learnable gain $g$ and bias $b$ have shape `(H,)`**: one parameter per neuron shared across all examples and time steps, allowing each neuron to learn its own target scale and baseline shift after normalization.


## Why LayerNorm stabilizes RNNs & Invariance Properties (Sections 3.1 & 5.1)

- **Scale cancellation across time steps**: If $a_t = [10, 100]$ (`mu = 55, sigma = 45`, normalized `[-1, +1]`) is scaled $10\times$ to $[100, 1000]$, then $\mu_t = 550$ and $\sigma_t = 450$ also scale by $10\times$, so $(a_t - \mu_t)/\sigma_t$ stays `[-1, +1]`. In backward pass, scaling $W_{hh}$ by $\alpha$ is cancelled at every step by $1/\sigma_t \propto 1/\alpha$ in $\partial \text{LN}(a_t)/\partial a_t$.
- **Row-vs-Column Invariance Duality (Table 1, p. 4)**:
  - **BatchNorm (column-wise across `B`)**: invariant to scaling a single neuron's weight vector $w_i$, but *not* invariant to scaling a single training example $x$.
  - **LayerNorm (row-wise across `H`)**: invariant to scaling the full weight matrix $W$, shifting all columns of $W$ by $\mathbf{1}\gamma^\top$, or scaling a single training example $x$, but *not* invariant to scaling only one neuron's weight vector $w_i$.


# Implementation


## 1. Setup


In [ ]:
import random

import torch
from torch import nn

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
seed = 42
random.seed(seed)
torch.manual_seed(seed)
print(f"Using device: {device}")

## 2. LayerNorm from scratch


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-5) -> None:
        super().__init__()
        # initialize learnable gain g (ones) and bias b (zeros) of shape (H,)
        self.hidden_size = hidden_size
        self.eps = eps
        self.g = nn.Parameter(torch.ones(self.hidden_size))
        self.b = nn.Parameter(torch.zeros(self.hidden_size))

    def forward(self, a: torch.Tensor) -> torch.Tensor:
        # compute mu and sigma across the last dimension (-1, size H) with keepdim=True,
        # assert shapes, and return g * (a - mu) / sigma + b
        mu = torch.mean(a, dim=-1, keepdim=True)
        assert mu.shape == (*a.shape[:-1], 1)
        var = torch.var(a, dim=-1, keepdim=True, unbiased=False)
        sigma = torch.sqrt(var + self.eps)
        assert sigma.shape == (*a.shape[:-1], 1)
        return self.g * (a - mu)/sigma + self.b

In [ ]:
# Sanity check LayerNorm on a = [[10., 100.], [20., 200.], [30., 300.]]
# and verify torch.allclose(ln(a), ln(10.0 * a))
a = torch.tensor([[10., 100.], [20., 200.], [30., 300.]])
ln = LayerNorm(2)
print(ln(10*a))
assert torch.allclose(ln(a), ln(10.0 * a), atol=1e-5)

## 3. LayerNorm RNN Cell (Paper Section 3, Eq. 5)


In [ ]:
class RNNCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, use_ln: bool = True) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.use_ln = use_ln
        self.W_xh = nn.Linear(self.input_size, self.hidden_size, bias=False)
        self.W_hh = nn.Linear(self.hidden_size, self.hidden_size, bias=not use_ln)
        if self.use_ln:
            self.ln = LayerNorm(self.hidden_size)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        # x_t: (B, D), h_prev: (B, H) -> returns h_t: (B, H)
        # Paper Eq. 5: a_t = W_hh h_{t-1} + W_xh x_t; h_t = tanh(LN(a_t)) if use_ln else tanh(a_t)
        a_t = self.W_hh(h_prev) + self.W_xh(x_t)
        assert a_t.shape == (x_t.shape[0], self.hidden_size)
        h_t = torch.tanh(self.ln(a_t)) if self.use_ln else torch.tanh(a_t)
        assert h_t.shape == (x_t.shape[0], self.hidden_size)
        return h_t

## 4. Experiment 1: Gradient & activation stability across time steps (Section 3.1)


In [ ]:
def unroll_and_measure_grads(
    cell: RNNCell, x: torch.Tensor, weight_scale: float = 1.0
) -> list[float]:
    """Unroll `cell` across T steps on `x` (B, T, D), backprop from final h_T, and return ||dL/dh_t|| for t=0..T-1."""
    B, T, _ = x.shape
    H = cell.hidden_size
    with torch.no_grad():
        cell.W_hh.weight.mul_(weight_scale)
        cell.W_xh.weight.mul_(weight_scale)
    h = torch.zeros(B, H, device=x.device)
    states: list[torch.Tensor] = []
    for t in range(T):
        h = cell(x[:, t, :], h)
        h.retain_grad()
        states.append(h)
    loss = states[-1].sum()
    loss.backward()
    return [s.grad.norm().item() for s in states if s]

In [ ]:
torch.manual_seed(42)
x = torch.randn(4, 50, 16)
for scale in (0.25, 1.0, 2.5):
    torch.manual_seed(42)
    g_vanilla = unroll_and_measure_grads(RNNCell(16, 32, use_ln=False), x, weight_scale=scale)
    torch.manual_seed(42)
    g_ln = unroll_and_measure_grads(RNNCell(16, 32, use_ln=True), x, weight_scale=scale)
    print(
        f"scale={scale:4.2f} | Vanilla ||dL/dh_0||={g_vanilla[0]:.3e} (t=49: {g_vanilla[-1]:.2f}) "
        f"| LayerNorm ||dL/dh_0||={g_ln[0]:.3e} (t=49: {g_ln[-1]:.2f})"
    )

## 5. Experiment 2: Remembering token t=0 across T=10 steps

When initial recurrent weights are small (`init_scale = 0.25`), a Vanilla RNN suffers from vanishing activations and gradients ($0.25^{10} \approx 10^{-6}$), stalling on a loss plateau for the first ~70 steps, whereas LayerNorm resets activation variance to `1.0` at every step and learns immediately.


In [13]:
class SequenceRegressor(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, use_ln: bool) -> None:
        super().__init__()
        self.cell = RNNCell(input_size, hidden_size, use_ln=use_ln)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        h = torch.zeros(B, self.cell.hidden_size, device=x.device)
        for t in range(T):
            h = self.cell(x[:, t, :], h)
        return self.head(h).squeeze(-1)


def train_toy_task(use_ln: bool, steps: int = 150, lr: float = 3e-3, init_scale: float = 0.25) -> list[float]:
    torch.manual_seed(42)
    model = SequenceRegressor(input_size=8, hidden_size=32, use_ln=use_ln)
    with torch.no_grad():
        model.cell.W_hh.weight.mul_(init_scale)
        model.cell.W_xh.weight.mul_(init_scale)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []
    torch.manual_seed(999)
    X_data = torch.zeros(512, 10, 8)
    X_data[:, 0, :] = torch.randn(512, 8)
    y_data = X_data[:, 0, :4].sum(dim=1)
    for _ in range(steps):
        pred = model(X_data)
        loss = nn.functional.mse_loss(pred, y_data)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


losses_vanilla = train_toy_task(use_ln=False)
losses_ln = train_toy_task(use_ln=True)
print(f"Vanilla MSE  : step 0={losses_vanilla[0]:.4f} | step 30={losses_vanilla[29]:.4f} | step 60={losses_vanilla[59]:.4f}")
print(f"LayerNorm MSE: step 0={losses_ln[0]:.4f} | step 30={losses_ln[29]:.4f} | step 60={losses_ln[59]:.4f}")

Vanilla MSE  : step 0=3.9889 | step 30=3.9816 | step 60=3.9815
LayerNorm MSE: step 0=4.2255 | step 30=0.3343 | step 60=0.1918
